# EXP-2026-001 on Colab: Stage 3 checks on the real model

Every check calls tested code in `rhuh_synthesis/stage3.py`:
1. The released MedSegFactory checkpoint loads **strictly** into the vendored 4-channel UNet.
2. Expanding `conv_in` 4 -> 11 leaves outputs unchanged at zero added weights (64 x 64 latents = the selected 512 px grid).
3. Memory and speed of one full training step for several microbatch sizes, bf16 and fp32.
4. One real-data step on two training patients: finite loss, gradients in the adapter and JCA, frozen VAE/CLIP, and a checkpoint save/reload round trip.
5. A short run of the four-channel sampler and the color-channel mask decoder on the real model.

This is not training. Downloads: the Stable Diffusion 1.5 components (license accepted) and the MedSegFactory weights from Hugging Face `JohnWeck/StableDiffusion`. The notebook prints that repository's license field: check its terms. Nothing needs uploading. Use the GPU runtime. A summary is written once to `synthesis/EXP-2026-001/stage3/` on Drive.

**Clear all outputs before saving, sharing or committing this notebook.**

In [ ]:
REPO_URL = "https://github.com/AIVIETNAM-AIO-Tuan/MedSegFactory.git"  # your fork only
BRANCH = "exp-2026-001-implementation"
DRIVE_RHUH = "/content/drive/MyDrive/RHUH-GBM"
WORK = "/content/MedSegFactory"
SD_REPO = "stable-diffusion-v1-5/stable-diffusion-v1-5"
MSF_REPO, MSF_FILE = "JohnWeck/StableDiffusion", "checkpoint-300.pth"
MICROBATCHES = [1, 2, 4, 6, 8]   # slice pairs per microbatch (each pair = image + mask sample)
PRECISIONS = ["bf16", "fp32"]
REAL_STEP_MICROBATCH = 2

## 1. Mount Drive, get the latest code from the fork, link the data

In [ ]:
import copy
import json
import os
import subprocess
import sys
import time
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
if not Path(WORK).is_dir():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, WORK], check=True)
subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)
link = Path(WORK, "data", "RHUH-GBM")
link.parent.mkdir(exist_ok=True)
if not link.exists():
    link.symlink_to(DRIVE_RHUH, target_is_directory=True)
os.chdir(WORK)
sys.path.insert(0, WORK)
PRIVATE = Path("data/RHUH-GBM/synthesis/EXP-2026-001")
assert (PRIVATE / "split/split.json").exists(), "The synthesis folder is missing on Drive"
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "diffusers==0.40.0", "transformers==5.17.0", "accelerate==1.15.0",
                "nibabel>=5.3,<6", "SimpleITK==2.5.6", "pydicom>=3,<4", "scikit-image>=0.25"], check=True)
commit = subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip()
print(commit)
summary = {"commit": commit}

## 2. Download the MedSegFactory checkpoint and load it strictly

In [ ]:
import torch
from huggingface_hub import hf_hub_download, model_info

from rhuh_synthesis.manifest import file_sha256
from rhuh_synthesis.stage3 import load_reference_unet
from StableDiffusion.Our_UNet import UNet2DConditionModel

try:
    card = model_info(MSF_REPO).card_data
    license_field = getattr(card, "license", None) if card else None
except Exception as exc:
    license_field = f"could not read model info: {exc}"
print("MedSegFactory weights:", MSF_REPO, "| license field:", license_field)
checkpoint = hf_hub_download(MSF_REPO, MSF_FILE)
unet_config = UNet2DConditionModel.load_config(SD_REPO, subfolder="unet")
unet4, load_report = load_reference_unet(UNet2DConditionModel, unet_config, checkpoint)
summary["checkpoint"] = {"repo": MSF_REPO, "file": MSF_FILE, "sha256": file_sha256(checkpoint),
                         "license_field": str(license_field), **load_report}
print({k: (v[:10] if isinstance(v, list) else v) for k, v in summary["checkpoint"].items()})
assert unet4 is not None, "Checkpoint keys do not match the vendored UNet: stop and report"


## 3. Frozen components and schedulers

In [ ]:
from diffusers import AutoencoderKL, DDPMScheduler
from transformers import CLIPTextModel, CLIPTokenizer

from rhuh_synthesis.model import reference_ddim_scheduler

vae = AutoencoderKL.from_pretrained(SD_REPO, subfolder="vae").to("cuda").eval().requires_grad_(False)
tokenizer = CLIPTokenizer.from_pretrained(SD_REPO, subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(SD_REPO, subfolder="text_encoder").to("cuda").eval().requires_grad_(False)
noise_scheduler = DDPMScheduler.from_pretrained(SD_REPO, subfolder="scheduler")
print("VAE scaling factor:", vae.config.scaling_factor, "| CLIP max tokens:", tokenizer.model_max_length)

## 4. Adapter equivalence on the real model (64 x 64 latents)

In [ ]:
from rhuh_synthesis.stage3 import adapter_equivalence

summary["equivalence"] = adapter_equivalence(unet4, latent_hw=(64, 64), cross_dim=768, device="cuda")
print(summary["equivalence"])

## 5. Memory and speed of one training step (random inputs, throwaway copies)

In [ ]:
import pandas as pd

from rhuh_synthesis.model import expand_conv_in
from rhuh_synthesis.stage3 import profile_training

rows = []
for precision in PRECISIONS:
    trial = expand_conv_in(copy.deepcopy(unet4), 11)
    rows += profile_training(trial, noise_scheduler, microbatches=MICROBATCHES, precision=precision,
                             latent_hw=(64, 64), cross_dim=768, device="cuda")
    del trial
    torch.cuda.empty_cache()
summary["profile"] = rows
pd.DataFrame(rows)

## 6. One real-data step, checkpoint round trip and sampler check

In [ ]:
import numpy as np

from rhuh_synthesis.stage3 import checkpoint_roundtrip, clip_encoder, real_data_step, sampler_check
from rhuh_synthesis.training import SliceSampler, build_feature_cache, slice_pools
from rhuh_synthesis.working_grid import to_working_grid

config = json.loads(Path("configs/experiments/EXP-2026-001.json").read_text(encoding="utf-8"))
manifest = json.loads((PRIVATE / "manifest/case_manifest.json").read_text(encoding="utf-8"))["patients"]
split = json.loads((PRIVATE / "split/split.json").read_text(encoding="utf-8"))
policy = json.loads((PRIVATE / "registration_policy/acceptance_policy_v1.json").read_text(encoding="utf-8"))
train = [p for p in split["roles"]["train"]["patients"] if p not in policy["excluded_training"]]
chosen = [next(p for p in train if manifest[p]["post_id3_present"]),
          next(p for p in train if not manifest[p]["post_id3_present"])]
grids = {}
for patient in chosen:
    with np.load(next(PRIVATE.glob(f"export_v1/train*/{patient}.npz"))) as data:
        arrays = {key: data[key] for key in data.files}
    _, grids[patient] = to_working_grid(arrays)
start = time.perf_counter()
cache = build_feature_cache(grids, vae, clip_encoder(tokenizer, text_encoder, "cuda"), config["text"],
                            batch=8, device="cuda", mask_codec=config["representation"]["mask_codec"])
print(f"feature cache for 2 patients in {time.perf_counter() - start:.0f}s")
sampler = SliceSampler({p: slice_pools(grids[p]["label"]) for p in chosen}, seed=config["training"]["train_seed"])
rng, generator = np.random.default_rng(2026), torch.Generator().manual_seed(2026)
unet11 = expand_conv_in(copy.deepcopy(unet4), 11).to("cuda")
step_report, optimizer = real_data_step(unet11, cache, sampler, noise_scheduler, microbatch=REAL_STEP_MICROBATCH,
                                        rng=rng, generator=generator, precision="bf16", frozen=(vae, text_encoder))
fresh = expand_conv_in(copy.deepcopy(unet4), 11).to("cuda")
path = Path("/content/stage3_checkpoint.pt")
roundtrip = checkpoint_roundtrip(path, unet11=unet11, optimizer=optimizer, sampler=sampler, rng=rng,
                                 generator=generator, fresh_unet11=fresh, config_sha256="stage3-check")
path.unlink()
del fresh, optimizer
torch.cuda.empty_cache()
positive_slice = int(np.argmax((grids[chosen[0]]["label"] == 3).sum(axis=(1, 2))))
check = sampler_check(unet11, vae, reference_ddim_scheduler(), cache, chosen[0], positive_slice, steps=20,
                      guidance_scale=7.5, generator=torch.Generator().manual_seed(11),
                      mask_codec=config["representation"]["mask_codec"], device="cuda")
summary["real_step"] = step_report
summary["checkpoint_roundtrip"] = roundtrip
summary["sampler_check"] = check
print({k: v for k, v in step_report.items() if k != "items"})
print(roundtrip)
print(check)

## 7. Save the summary (write-once) and print the part to send

In [ ]:
from rhuh_synthesis.manifest import write_once_json

summary["peak_gib_total"] = torch.cuda.max_memory_allocated() / 2**30
stamp = time.strftime("%Y%m%dT%H%M%S")
path, digest = write_once_json(PRIVATE / "stage3" / f"summary_{stamp}.json", summary)
print("saved", path, digest[:12])
shareable = {
    "commit": summary["commit"],
    "checkpoint": {k: summary["checkpoint"][k] for k in ("sha256", "license_field", "tensors_in_checkpoint",
                                                          "jca_tensors", "parameters")},
    "missing/unexpected keys": [len(summary["checkpoint"]["missing_keys"]), len(summary["checkpoint"]["unexpected_keys"])],
    "equivalence": summary["equivalence"],
    "profile": [{k: r.get(k) for k in ("precision", "microbatch_pairs", "oom", "peak_gib", "seconds_per_step")}
                for r in summary["profile"]],
    "real_step": {k: v for k, v in summary["real_step"].items() if k != "items"},
    "checkpoint_roundtrip": summary["checkpoint_roundtrip"],
    "sampler_check": summary["sampler_check"],
}
print(json.dumps(shareable, indent=1))

## Next

Send Claude the printed JSON block (no patient IDs). It decides the microbatch and precision for Stage 4, the overfit check on about 16 training slices. Stage 4 needs your explicit go-ahead.